# 07 · NeuroSAT-Guided MiniSat

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

In [ ]:
!pip install python-sat -q

In [ ]:
import pickle
import random
import time
import torch
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from pysat.solvers import Minisat22

from neurosat import NeuroSAT, make_batch, to_device

## Settings

In [ ]:
SEED = 42
DIM = 128
ROUNDS = 128
HARD_VARS = [100, 150]
HARD_PAIRS = 50

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load model

In [ ]:
model = NeuroSAT(DIM).to(device)
model.load_state_dict(torch.load("neurosat.pt", map_location=device))
model.eval()

## Harder SR(n) problems

In [ ]:
def clause_size():
    k = 2
    if random.random() < 0.3:
        k += 1
    while random.random() > 0.4:
        k += 1
    return k


def random_clause(n_vars):
    k = min(clause_size(), n_vars)
    variables = random.sample(range(1, n_vars + 1), k)
    return [v if random.random() < 0.5 else -v for v in variables]


def generate_pair(n_vars):
    clauses = []
    solver = Minisat22()

    while True:
        clause = random_clause(n_vars)
        solver.add_clause(clause)
        if solver.solve():
            clauses.append(clause)
        else:
            break

    solver.delete()

    unsat_clauses = clauses + [clause]

    flipped = clause.copy()
    i = random.randrange(len(flipped))
    flipped[i] = -flipped[i]
    sat_clauses = clauses + [flipped]

    return sat_clauses, unsat_clauses


def make_dataset(n_pairs, n_vars):
    data = []
    for _ in range(n_pairs):
        sat_clauses, unsat_clauses = generate_pair(n_vars)
        data.append({"n_vars": n_vars, "clauses": sat_clauses, "label": 1})
        data.append({"n_vars": n_vars, "clauses": unsat_clauses, "label": 0})
    return data

## Datasets

In [ ]:
with open("test_large.pkl", "rb") as f:
    large_test_data = pickle.load(f)

datasets = {}
for n, data in large_test_data.items():
    datasets[str(n)] = data

for n in HARD_VARS:
    datasets[str(n)] = make_dataset(HARD_PAIRS, n)

for name, data in datasets.items():
    print(f"n={name}: {len(data)} formulas")

## MiniSat with phase hints

In [ ]:
def run_minisat(clauses, phases=None):
    with Minisat22(bootstrap_with=clauses) as solver:
        if phases is not None:
            solver.set_phases(phases)
        start = time.perf_counter()
        solver.solve()
        elapsed = time.perf_counter() - start
        stats = solver.accum_stats()
    return {
        "time": elapsed,
        "decisions": stats["decisions"],
        "conflicts": stats["conflicts"],
        "propagations": stats["propagations"],
    }

## Phases

In [ ]:
def count_satisfied(clauses, assignment):
    return sum(any(assignment[abs(lit) - 1] == (lit > 0) for lit in clause) for clause in clauses)


def random_phases(problem):
    return [v if random.random() < 0.5 else -v for v in range(1, problem["n_vars"] + 1)]


def neurosat_phases(problem):
    start = time.perf_counter()

    batch = to_device(make_batch([problem]), device)
    with torch.no_grad():
        _, L_h = model(batch, ROUNDS, return_literals=True)

    clusters = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(L_h.cpu().numpy())
    n = problem["n_vars"]

    best = None
    best_score = -1
    for true_cluster in [0, 1]:
        assignment = [clusters[i] == true_cluster for i in range(n)]
        score = count_satisfied(problem["clauses"], assignment)
        if score > best_score:
            best = assignment
            best_score = score

    phases = [v + 1 if best[v] else -(v + 1) for v in range(n)]
    elapsed = time.perf_counter() - start
    solved = best_score == len(problem["clauses"])
    return phases, elapsed, solved

## Quick check

In [ ]:
problem = datasets["100"][0]
phases, elapsed, solved = neurosat_phases(problem)

print("Default: ", run_minisat(problem["clauses"]))
print("Random:  ", run_minisat(problem["clauses"], random_phases(problem)))
print("NeuroSAT:", run_minisat(problem["clauses"], phases))
print(f"NeuroSAT time {elapsed * 1000:.1f} ms, assignment already satisfies all clauses: {solved}")

## Run

In [ ]:
METHODS = ["default", "random", "neurosat"]
METRICS = ["time", "decisions", "conflicts", "propagations"]


def mean(values):
    return sum(values) / len(values)


results = {}

for name, data in datasets.items():
    results[name] = {}
    for label, label_name in [(1, "sat"), (0, "unsat")]:
        problems = [p for p in data if p["label"] == label]
        runs = {m: [] for m in METHODS}
        neural_times = []
        neural_solved = []

        for problem in problems:
            phases, elapsed, solved = neurosat_phases(problem)
            neural_times.append(elapsed)
            neural_solved.append(solved)

            runs["default"].append(run_minisat(problem["clauses"]))
            runs["random"].append(run_minisat(problem["clauses"], random_phases(problem)))
            runs["neurosat"].append(run_minisat(problem["clauses"], phases))

        summary = {}
        for method in METHODS:
            summary[method] = {metric: mean([r[metric] for r in runs[method]]) for metric in METRICS}
        summary["neural_time"] = mean(neural_times)
        summary["neural_solved"] = mean(neural_solved)
        results[name][label_name] = summary

    print(f"Done n={name}")

## Results table

In [ ]:
for label_name in ["sat", "unsat"]:
    print(label_name.upper())
    print(f"{'n':>5s} | {'method':>8s} | {'decisions':>10s} | {'conflicts':>10s} | {'solve ms':>9s}")
    for name, r in results.items():
        for method in METHODS:
            s = r[label_name][method]
            print(f"{name:>5s} | {method:>8s} | {s['decisions']:10.1f} | {s['conflicts']:10.1f} | {s['time'] * 1000:9.2f}")
        print(f"{name:>5s} | {'network':>8s} | {'':>10s} | {'':>10s} | {r[label_name]['neural_time'] * 1000:9.2f}")
    print()

print("Share of SAT formulas already solved by the NeuroSAT assignment:")
for name, r in results.items():
    print(f"  n={name}: {r['sat']['neural_solved']:.3f}")

## Plots

In [ ]:
names = list(results.keys())
x = range(len(names))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, label_name in zip(axes, ["sat", "unsat"]):
    for k, method in enumerate(METHODS):
        values = [results[n][label_name][method]["conflicts"] + 1 for n in names]
        ax.bar([i + (k - 1) * width for i in x], values, width, label=method)
    ax.set_xticks(list(x))
    ax.set_xticklabels(names)
    ax.set_yscale("log")
    ax.set_xlabel("Variables")
    ax.set_ylabel("Mean conflicts + 1")
    ax.set_title(label_name.upper())
    ax.legend()

plt.tight_layout()
plt.savefig("guided_minisat.png", dpi=150)
plt.show()

## Save

In [ ]:
with open("guided_minisat_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved guided_minisat_results.pkl and guided_minisat.png")